# Chapter 7: Reshaping Data — Pivot, Melt, Stack

Data often needs to be reshaped between **wide** and **long** formats.

1. `pivot_table` — long → wide (with aggregation)
2. `melt` — wide → long
3. `stack` / `unstack`
4. `pd.crosstab`
5. Practical example: survey data reshaping

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

## 7.1 `pivot_table` — Long to Wide with Aggregation

In [ ]:
# Sales data (long format)
sales = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=120, freq='D'),
    'region': np.random.choice(['North', 'South', 'East', 'West'], 120),
    'product': np.random.choice(['Widget', 'Gadget', 'Doohickey'], 120),
    'revenue': np.random.randint(100, 1000, 120),
})
sales['month'] = sales['date'].dt.month_name()
sales.head()

In [ ]:
# Basic pivot table
pt = pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='product',
    aggfunc='sum'
)
pt

In [ ]:
# Multiple aggregation functions
pt_multi = pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='product',
    aggfunc=['sum', 'mean', 'count']
)
pt_multi

In [ ]:
# Add margins (row & column totals)
pt_margins = pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='product',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
pt_margins

## 7.2 `melt` — Wide to Long

`melt` is the inverse of pivot — it "unpivots" a wide table into long format.

In [ ]:
# Wide format: test scores across subjects
wide = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie'],
    'math': [90, 85, 78],
    'science': [88, 92, 80],
    'english': [95, 80, 85],
})
print('Wide format:')
wide

In [ ]:
# Melt to long format
long = pd.melt(
    wide,
    id_vars='student',
    value_vars=['math', 'science', 'english'],
    var_name='subject',
    value_name='score'
)
print('Long format:')
long

## 7.3 `stack` & `unstack`

- `stack()` — pivots columns → rows (wide → long)
- `unstack()` — pivots rows → columns (long → wide)

In [ ]:
# Start with a multi-indexed Series
s = sales.groupby(['region', 'product'])['revenue'].sum()
print('Multi-indexed Series:')
print(s)
print(f'\nShape: {s.shape}')

In [ ]:
# unstack — pivot innermost index level to columns
unstacked = s.unstack()
print('Unstacked (wide):')
unstacked

In [ ]:
# stack — reverse operation
restacked = unstacked.stack()
print('Re-stacked (long):')
print(restacked.head(8))

## 7.4 `pd.crosstab` — Frequency Tables

In [ ]:
# Simple frequency table
ct = pd.crosstab(sales['region'], sales['product'])
ct

In [ ]:
# With aggregation (like pivot_table)
ct_sum = pd.crosstab(
    sales['region'], sales['product'],
    values=sales['revenue'], aggfunc='sum',
    margins=True
)
ct_sum

In [ ]:
# Normalize to percentages
ct_pct = pd.crosstab(
    sales['region'], sales['product'],
    normalize='index'  # row percentages (sum to 1 per row)
).round(3)
ct_pct

## 7.5 Practical Example: Survey Data Reshaping

In [ ]:
# Simulated survey data (wide format)
survey_wide = pd.DataFrame({
    'respondent_id': range(1, 6),
    'age': [25, 34, 45, 29, 52],
    'q1_satisfaction': [4, 5, 3, 4, 2],
    'q2_likelihood': [3, 5, 4, 5, 1],
    'q3_quality': [5, 4, 3, 4, 3],
})
print('Wide survey data:')
survey_wide

In [ ]:
# Melt to long format for analysis
survey_long = pd.melt(
    survey_wide,
    id_vars=['respondent_id', 'age'],
    value_vars=['q1_satisfaction', 'q2_likelihood', 'q3_quality'],
    var_name='question',
    value_name='rating'
)
print('Long survey data:')
survey_long

In [ ]:
# Analyze: average rating per question
survey_long.groupby('question')['rating'].agg(['mean', 'std', 'count']).round(2)

In [ ]:
# Pivot back to wide (full circle!)
back_to_wide = survey_long.pivot_table(
    index=['respondent_id', 'age'],
    columns='question',
    values='rating'
).reset_index()
back_to_wide

## 📝 Key Takeaways

| Operation | Direction | Use Case |
|-----------|-----------|----------|
| `pivot_table` | Long → Wide | Summarize with aggregation |
| `melt` | Wide → Long | Normalize for analysis |
| `stack` | Columns → Rows | Multi-index reshaping |
| `unstack` | Rows → Columns | Multi-index reshaping |
| `crosstab` | Frequency table | Quick cross-tabulation |